# ClimaTrend: Weather Analysis and Forecasting

Welcome to the **ClimaTrend** notebook. This notebook demonstrates how to load, clean, analyze, and forecast historical weather data for any location. It uses Open-Meteo for meteorological data, implements seasonal decomposition, and compares **Prophet** against a baseline **Machine Learning Regressor**.

In [ ]:
# Add parent directory to path to allow importing src
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_acquisition import geocode_city, fetch_historical_weather
from src.data_cleaning import process_pipeline
from src.eda import (
    generate_summary_statistics,
    perform_seasonal_decomposition,
    compute_acf_pacf,
    calculate_correlations
)
from src.forecasting_model import (
    train_test_split_ts,
    fit_forecast_prophet,
    fit_forecast_ml
)
from src.chart_visualizations import (
    plot_time_series,
    plot_seasonal_decomposition,
    plot_acf_pacf,
    plot_correlation_heatmap,
    plot_monthly_distribution,
    plot_wind_rose
)

## 1. Data Acquisition
We start by geocoding a city (e.g., London) to retrieve coordinates and fetch historical hourly weather data for the last 3 years.

In [ ]:
location = "London"
geo_info = geocode_city(location)
if geo_info:
    lat, lon = geo_info['lat'], geo_info['lon']
    print(f"Coordinates: {lat}, {lon} ({geo_info['country']})")
else:
    # Default fallback
    lat, lon = 51.5074, -0.1278
    print("Using default coordinates for London")

raw_data = fetch_historical_weather(lat, lon, "2023-01-01", "2025-12-31", city_name=location)
raw_data.head()

## 2. Data Cleaning & Aggregation
We process the raw hourly data through the cleaning pipeline to impute missing values, cap outliers, and aggregate to daily intervals.

In [ ]:
df_hourly, df_daily = process_pipeline(raw_data)
print("Hourly cleaned shape:", df_hourly.shape)
print("Daily aggregated shape:", df_daily.shape)
df_daily.head()

## 3. Exploratory Data Analysis (EDA)
Let's look at the summary statistics, correlations, and distributions of temperature.

In [ ]:
summary = generate_summary_statistics(df_daily)
summary

In [ ]:
# Plot Temperature Trend
fig_ts = plot_time_series(df_daily, "temperature_2m_mean", "Daily Average Temperature")
plt.show()

In [ ]:
# Plot Correlation Heatmap
corr_matrix = calculate_correlations(df_daily)
fig_corr = plot_correlation_heatmap(corr_matrix)
plt.show()

In [ ]:
# Monthly distributions (Violin Plot)
fig_dist = plot_monthly_distribution(df_daily, "temperature_2m_mean")
plt.show()

In [ ]:
# Wind Rose Plot
fig_wind = plot_wind_rose(df_daily)
plt.show()

## 4. Time-Series Decomposition & Autocorrelation
We extract the underlying trend, annual seasonality, and noise residuals from our temperature data, and analyze ACF/PACF values.

In [ ]:
obs, trend, seasonal, resid = perform_seasonal_decomposition(df_daily, "temperature_2m_mean")
fig_decomp = plot_seasonal_decomposition(obs, trend, seasonal, resid)
plt.show()

In [ ]:
acf_v, pacf_v = compute_acf_pacf(df_daily, "temperature_2m_mean")
fig_acf = plot_acf_pacf(acf_v, pacf_v)
plt.show()

## 5. Forecasting
We split the dataset into train and test sets, evaluate performance on the last 30 days of data, and generate a 90-day forecast.

In [ ]:
train_df, test_df = train_test_split_ts(df_daily, test_size_days=30)
forecast_horizon = 90
target_col = "temperature_2m_mean"

In [ ]:
# Prophet Model
try:
    prophet_forecast, prophet_metrics = fit_forecast_prophet(
        train_df, test_df, forecast_days=forecast_horizon, target_col=target_col
    )
    print("Prophet Metrics:", prophet_metrics)
except Exception as e:
    print("Prophet training failed/skipped:", e)

In [ ]:
# Random Forest baseline with recursive lags
rf_forecast, rf_metrics = fit_forecast_ml(
    train_df, test_df, forecast_days=forecast_horizon, target_col=target_col, model_type="random_forest"
)
print("Random Forest Metrics:", rf_metrics)

## Conclusion
This completes the meteorological time series modeling. You can launch the full web application by running `streamlit run climatrend/app.py` in your shell.